# Thành viên 2 – Feature Engineering
GROUP BY & Aggregation trên các bảng phụ: bureau, previous_application, installments, credit_card, POS_CASH.

In [1]:
import sys
sys.path.append('../src')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder \
    .appName('HomeCreditFeatures') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

ModuleNotFoundError: No module named 'pyspark'

## 1. Load bảng phụ

In [ ]:
CLEANED = '../dataset/cleaned'

bureau          = spark.read.parquet(f'{CLEANED}/bureau_cleaned.parquet')
bureau_balance  = spark.read.parquet(f'{CLEANED}/bureau_bal_cleaned.parquet')
prev            = spark.read.parquet(f'{CLEANED}/prev_app_cleaned.parquet')
inst            = spark.read.parquet(f'{CLEANED}/installments_cleaned.parquet')
cc              = spark.read.parquet(f'{CLEANED}/credit_card_cleaned.parquet')
pos             = spark.read.parquet(f'{CLEANED}/pos_cash_cleaned.parquet')

for name, df in [('bureau',bureau),('prev_app',prev),('installments',inst),('credit_card',cc),('pos_cash',pos)]:
    print(f'{name}: {df.count():,} rows')


## 2. Aggregate Bureau

In [ ]:
from feature_engineering import aggregate_bureau, aggregate_bureau_balance

agg_bb = aggregate_bureau_balance(bureau_balance)
print('Bureau balance aggregated:'); agg_bb.show(3)

agg_bureau = aggregate_bureau(bureau)
print('Bureau aggregated:'); agg_bureau.show(3)

## 3. Aggregate Previous Application

In [ ]:
from feature_engineering import aggregate_previous_application

agg_prev = aggregate_previous_application(prev)
agg_prev.show(3)

# Visualise tỉ lệ approved vs refused
status_pd = prev.groupBy('NAME_CONTRACT_STATUS').count().toPandas()
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(status_pd['NAME_CONTRACT_STATUS'], status_pd['count'], color='teal')
ax.set_title('Previous Application – Contract Status')
ax.set_xlabel('Status'); ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/EDA_visualizations/prev_app_status.png', dpi=150)
plt.show()

## 4. Aggregate Installments, Credit Card, POS CASH

In [ ]:
from feature_engineering import aggregate_installments, aggregate_credit_card, aggregate_pos_cash

agg_inst = aggregate_installments(inst)
agg_cc   = aggregate_credit_card(cc)
agg_pos  = aggregate_pos_cash(pos)

for name, df in [('installments',agg_inst),('credit_card',agg_cc),('pos_cash',agg_pos)]:
    print(f'\n{name} ({df.count():,} rows):')
    df.show(3)

## 5. Thêm Đặc trưng Tài chính & Chuẩn bị ML
Tạo Credit-to-Income, Annuity-to-Income, VectorAssembler và StandardScaler.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Doc app_train_cleaned.parquet tu member 1
app_train = spark.read.parquet(f'{CLEANED}/app_train_cleaned.parquet')

# Tinh toan chung ty le tai chinh tren thogn tin app
app_train = app_train.withColumn('Credit_to_Income', F.col('AMT_CREDIT') / F.col('AMT_INCOME_TOTAL')) \
                     .withColumn('Annuity_to_Income', F.col('AMT_ANNUITY') / F.col('AMT_INCOME_TOTAL'))

# Gom cac dac trung tu aggregate roi join vao app_train (Chi lay mau don gian thoi)
# Chon mot vai cot numeric quan trong tu app_train
num_cols = [c for c, t in app_train.dtypes if t in ('int', 'double') and c != 'TARGET' and c != 'SK_ID_CURR']

assembler = VectorAssembler(inputCols=num_cols, outputCol='features', handleInvalid='skip')
app_features = assembler.transform(app_train)

scaler = StandardScaler(inputCol='features', outputCol='scaled_features', withStd=True, withMean=True)
scaler_model = scaler.fit(app_features)
final_df = scaler_model.transform(app_features)

print('Tạo đặc trưng và VectorAssembler thành công!')
final_df.select('SK_ID_CURR', 'Credit_to_Income', 'Annuity_to_Income', 'scaled_features').show(3, truncate=False)

# Luu file parquet features
final_df.select('SK_ID_CURR', 'TARGET', 'scaled_features').write.mode('overwrite').parquet('../dataset/features/app_train_features.parquet')
print('Đã lưu tập features tại ../dataset/features/app_train_features.parquet')

## 5. Thêm Đặc trưng Tài chính & Chuẩn bị ML
Tạo Credit-to-Income, Annuity-to-Income, VectorAssembler và StandardScaler.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Doc app_train_cleaned.parquet tu member 1
app_train = spark.read.parquet(f'{CLEANED}/app_train_cleaned.parquet')

# Tinh toan chung ty le tai chinh tren thogn tin app
app_train = app_train.withColumn('Credit_to_Income', F.col('AMT_CREDIT') / F.col('AMT_INCOME_TOTAL')) \
                     .withColumn('Annuity_to_Income', F.col('AMT_ANNUITY') / F.col('AMT_INCOME_TOTAL'))

# Gom cac dac trung tu aggregate roi join vao app_train (Chi lay mau don gian thoi)
# Chon mot vai cot numeric quan trong tu app_train
num_cols = [c for c, t in app_train.dtypes if t in ('int', 'double') and c != 'TARGET' and c != 'SK_ID_CURR']

assembler = VectorAssembler(inputCols=num_cols, outputCol='features', handleInvalid='skip')
app_features = assembler.transform(app_train)

scaler = StandardScaler(inputCol='features', outputCol='scaled_features', withStd=True, withMean=True)
scaler_model = scaler.fit(app_features)
final_df = scaler_model.transform(app_features)

print('Tạo đặc trưng và VectorAssembler thành công!')
final_df.select('SK_ID_CURR', 'Credit_to_Income', 'Annuity_to_Income', 'scaled_features').show(3, truncate=False)

# Luu file parquet features
final_df.select('SK_ID_CURR', 'TARGET', 'scaled_features').write.mode('overwrite').parquet('../dataset/features/app_train_features.parquet')
print('Đã lưu tập features tại ../dataset/features/app_train_features.parquet')

## 5. Thêm Đặc trưng Tài chính & Chuẩn bị ML
Tạo Credit-to-Income, Annuity-to-Income, VectorAssembler và StandardScaler.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Doc app_train_cleaned.parquet tu member 1
app_train = spark.read.parquet(f'{CLEANED}/app_train_cleaned.parquet')

# Tinh toan chung ty le tai chinh tren thogn tin app
app_train = app_train.withColumn('Credit_to_Income', F.col('AMT_CREDIT') / F.col('AMT_INCOME_TOTAL')) \
                     .withColumn('Annuity_to_Income', F.col('AMT_ANNUITY') / F.col('AMT_INCOME_TOTAL'))

# Gom cac dac trung tu aggregate roi join vao app_train (Chi lay mau don gian thoi)
# Chon mot vai cot numeric quan trong tu app_train
num_cols = [c for c, t in app_train.dtypes if t in ('int', 'double') and c != 'TARGET' and c != 'SK_ID_CURR']

assembler = VectorAssembler(inputCols=num_cols, outputCol='features', handleInvalid='skip')
app_features = assembler.transform(app_train)

scaler = StandardScaler(inputCol='features', outputCol='scaled_features', withStd=True, withMean=True)
scaler_model = scaler.fit(app_features)
final_df = scaler_model.transform(app_features)

print('Tạo đặc trưng và VectorAssembler thành công!')
final_df.select('SK_ID_CURR', 'Credit_to_Income', 'Annuity_to_Income', 'scaled_features').show(3, truncate=False)

# Luu file parquet features
final_df.select('SK_ID_CURR', 'TARGET', 'scaled_features').write.mode('overwrite').parquet('../dataset/features/app_train_features.parquet')
print('Đã lưu tập features tại ../dataset/features/app_train_features.parquet')

In [ ]:
spark.stop()